In [9]:
%pip install nemosis

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from databricks.connect import DatabricksSession

# Replace '0000-000000-xxxxxx' with your actual Cluster ID
spark = DatabricksSession.builder.clusterId("79a5c5436aa3e2ee").getOrCreate()

INFO: loading DEFAULT profile from ~/.databrickscfg: host, token
🚀 Connection established to the remote cluster!


In [19]:
import os
from nemosis import dynamic_data_compiler  # <-- Updated import
import pandas as pd
from pyspark.sql import SparkSession
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.clusterId("79a5c5436aa3e2ee").getOrCreate()

from databricks.sdk import WorkspaceClient # <--- The new missing link
dbutils = WorkspaceClient().dbutils

dbutils.widgets.text("catalog", "workspace") 
dbutils.widgets.text("schema", "default") # You can change 'default' to your short name for testing

# 2. Now you can safely 'get' them
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# Set the context so all following Spark commands use the right place
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"✅ Context Set: {catalog}.{schema}")

# 1. Configuration for the Free Version
table_name = "DISPATCHPRICE" 
regions = ["NSW1"] 
cache_dir = "/tmp/nemosis_cache"

if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# 2. Updated Loop with a local cache fix
for year in range(2015, 2026):
    print(f"--- Processing Year: {year} ---")

    try:
        # Pull data using the correct compiler and local cache
        df_raw = dynamic_data_compiler(
            start_time=f"{year}/01/01 00:00:00", 
            end_time=f"{year}/12/31 23:55:00", 
            table_name="DISPATCHPRICE",
            raw_data_location=cache_dir # This fixes the 'does not exist' error
        )
        
        df_filtered = df_raw[df_raw['REGIONID'] == "NSW1"]
        spark_df = spark.createDataFrame(df_filtered)
        
        target_table = "energy_bronze"
        
        if year == 2015:
            spark_df.write.format("delta").mode("overwrite").saveAsTable("target_table")
        else:
            spark_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("target_table")
            
        print(f"Successfully saved {year}")
        
    except Exception as e:
        print(f"Error processing {year}: {e}")
    

INFO: loading DEFAULT profile from ~/.databrickscfg: host, token
INFO: loading DEFAULT profile from ~/.databrickscfg: host, token


/Users/quangthanhdong/miniforge3/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:71: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


SparkConnectGrpcException: RESOURCE_DOES_NOT_EXIST: No cluster found matching: 79a5c5436aa3e2ee

In [6]:
from pyspark.sql import functions as F

# Load the table
df = spark.table("default.energy_bronze")

# Filter for October 2023
october_23_df = df.filter(
    (F.year("SETTLEMENTDATE") == 2023) &
    (F.month("SETTLEMENTDATE") == 10)
)

# Show the results
display(october_23_df.orderBy("SETTLEMENTDATE"))

NameError: name 'spark' is not defined